# 01 — Exploratory Data Analysis
**Forest Carbon Stock Estimation — Nainital District, Uttarakhand**

This notebook covers:
- Study area overview and bounding box visualisation
- GEDI footprint spatial distribution
- AGBD statistics and distribution
- Sentinel-2 band preview
- Correlation between AGBD and spectral/terrain features

In [ ]:
import sys
sys.path.insert(0, '..')   # repo root

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import yaml
from pathlib import Path

with open('../config/config.yaml') as f:
    CFG = yaml.safe_load(f)

PATHS = CFG['paths']
BBOX  = CFG['study_area']['bbox']

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans'})
print('Setup complete.')

## 1. Study Area Bounding Box

In [ ]:
import folium

centre = [
    (BBOX['ymin'] + BBOX['ymax']) / 2,
    (BBOX['xmin'] + BBOX['xmax']) / 2
]
m = folium.Map(location=centre, zoom_start=9, tiles='CartoDB positron')

folium.Rectangle(
    bounds=[[BBOX['ymin'], BBOX['xmin']], [BBOX['ymax'], BBOX['xmax']]],
    color='#1a5c38', weight=2, fill=True, fill_opacity=0.08,
    tooltip='Nainital District Study Area'
).add_to(m)

m

## 2. GEDI Footprint Distribution

In [ ]:
train_csv = Path('..') / PATHS['training_csv']
df = pd.read_csv(train_csv)
print(f'Training samples: {len(df):,}')
df.describe()

In [ ]:
# Spatial distribution
m2 = folium.Map(location=centre, zoom_start=9, tiles='CartoDB positron')
sample = df.sample(min(3000, len(df)), random_state=42)

for _, row in sample.iterrows():
    norm = min(max(row['agbd'] / 300, 0), 1)
    color = f'#{int(255*(1-norm)):02x}{int(100+155*norm):02x}{int(50*(1-norm)):02x}'
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=2, color=color, fill=True, fill_opacity=0.7,
        popup=f"AGBD: {row['agbd']:.1f} t/ha"
    ).add_to(m2)

m2

## 3. AGBD Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram
axes[0].hist(df['agbd'], bins=50, color='#2e7d52', edgecolor='white', lw=0.5)
axes[0].axvline(df['agbd'].mean(),   color='red',    ls='--', lw=1.5, label=f'Mean={df["agbd"].mean():.1f}')
axes[0].axvline(df['agbd'].median(), color='orange', ls='--', lw=1.5, label=f'Median={df["agbd"].median():.1f}')
axes[0].set_xlabel('AGBD (t/ha)'); axes[0].set_ylabel('Count')
axes[0].set_title('AGBD Histogram'); axes[0].legend()

# Boxplot by elevation quartile
df['elev_q'] = pd.qcut(df['elevation'], q=4, labels=['Q1 Low','Q2','Q3','Q4 High'])
df.boxplot(column='agbd', by='elev_q', ax=axes[1], grid=False)
axes[1].set_title('AGBD by Elevation Quartile')
axes[1].set_xlabel('Elevation Quartile'); axes[1].set_ylabel('AGBD (t/ha)')

# AGBD vs NDVI
axes[2].scatter(df['NDVI'], df['agbd'], alpha=0.2, s=6, color='#4caf50')
axes[2].set_xlabel('NDVI'); axes[2].set_ylabel('AGBD (t/ha)')
axes[2].set_title('AGBD vs NDVI')

plt.suptitle('')
plt.tight_layout()
plt.show()

## 4. Feature Correlation Heatmap

In [ ]:
feature_cols = CFG['features']['model_features'] + ['agbd']
corr = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix (lower triangle)', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

# AGBD correlations sorted
print('Correlations with AGBD:')
print(corr['agbd'].drop('agbd').sort_values(ascending=False).to_string())

## 5. Sentinel-2 Band Preview (if available)

In [ ]:
s2_path = Path('..') / PATHS['sentinel_stack']

if s2_path.exists():
    with rasterio.open(s2_path) as src:
        # True colour: B4=R, B3=G, B2=B  (bands 3,2,1 in our stack)
        r = src.read(3)  # B4 Red
        g = src.read(2)  # B3 Green
        b = src.read(1)  # B2 Blue

    def normalise(band):
        p2, p98 = np.nanpercentile(band, [2, 98])
        return np.clip((band - p2) / (p98 - p2 + 1e-9), 0, 1)

    rgb = np.dstack([normalise(r), normalise(g), normalise(b)])

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(rgb)
    ax.set_title('Sentinel-2 True Colour — Nainital District', fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f'Sentinel-2 stack not found at {s2_path}.')
    print('Export from GEE first (src/data/gee_export.js).')